In [0]:
%sql
WITH
fd AS (
  SELECT DISTINCT
    cdv.category_name AS Category,
    cal.`PG Year (Real)`        AS `PG Year (Real)`,
    cal.`PG Quarter (Real)`     AS `PG Quarter (Real)`,
    cal.fy_half_year_name       AS fy_half_year_name,
    cjdv.`FN Code2`             AS `FN Code2`,
    cjdv.`FN Name2`             AS `FN Name2`
  FROM hive_metastore.japan_gold.category_dim_vw cdv
  CROSS JOIN (
    SELECT DISTINCT
      `PG Year (Real)`,
      `PG Quarter (Real)`,
      fy_half_year_name
    FROM hive_metastore.japan_linkedexcel.calender_dim_vw
    WHERE `PG Year (Real)` = 'FY2425'
      AND fy_half_year_name = '2H'
  ) cal
  CROSS JOIN (
    SELECT DISTINCT
      `FN Code2`,
      `FN Name2`
    FROM hive_metastore.japan_linkedexcel.customer_japan_dim_vw
    WHERE `FN Code2` IS NOT NULL
      AND `Customer Parent Group` = 'RT'
  ) cjdv
  WHERE cdv.category_name IS NOT NULL
    AND cdv.category_name NOT IN (
      'P&G non playing',
      'Super Premium Skin Care',
      'Household cleaning',
      'Female Skin Care',
      'Kitchen Cleaning'
    )
),

fact_union AS (

  /* ---------------- SHIP ---------------- */
  SELECT
    'SHIP' AS src,
    cal.`PG Year (Real)`        AS `PG Year (Real)`,
    cal.`PG Quarter (Real)`     AS `PG Quarter (Real)`,
    cal.fy_half_year_name       AS fy_half_year_name,
    cjdv.`FN Code2`             AS `FN Code2`,
    cjdv.`FN Name2`             AS `FN Name2`,
    pdv.`03.Category`           AS prod_category,
    pdv.`04.SubCategory (Lang)` AS subcat_lang,
    pdv.`76.Overlapping SKU (Electro)` AS overlapping_sku,
    cjdv.`Zakka Flag2`          AS zakka_flag2,
    cjdv.`Gillette Flag2`       AS gillette_flag2,
    CAST(NULL AS STRING)        AS wh_parent_group,
    SUM(CAST(sfv.gross_transact_amt AS DECIMAL(38,6))) AS giv,
    SUM(CAST(sfv.gross_transact_amt AS DECIMAL(38,6))) AS niv_calc
  FROM hive_metastore.japan_linkedexcel.shipment_fct_vw sfv
  JOIN hive_metastore.japan_linkedexcel.customer_japan_dim_vw cjdv
    ON cjdv.`Cust Key` = sfv.reporting_cust_key
  JOIN hive_metastore.japan_linkedexcel.product_dim_vw pdv
    ON pdv.prod_key = sfv.prod_key
  JOIN hive_metastore.japan_linkedexcel.calender_dim_vw cal
    ON sfv.day_date = cal.`Day Date (Real)`
  WHERE cal.`PG Year (Real)` = 'FY2425'
    AND cal.fy_half_year_name = '2H'
    AND cjdv.`FN Code2` IS NOT NULL
    AND cjdv.`Customer Parent Group` = 'RT'
  GROUP BY
    cal.`PG Year (Real)`,
    cal.`PG Quarter (Real)`,
    cal.fy_half_year_name,
    cjdv.`FN Code2`,
    cjdv.`FN Name2`,
    pdv.`03.Category`,
    pdv.`04.SubCategory (Lang)`,
    pdv.`76.Overlapping SKU (Electro)`,
    cjdv.`Zakka Flag2`,
    cjdv.`Gillette Flag2`

  UNION ALL

  /* ---------------- INDIRECT ---------------- */
  SELECT
    'IND' AS src,
    cal.`PG Year (Real)`        AS `PG Year (Real)`,
    cal.`PG Quarter (Real)`     AS `PG Quarter (Real)`,
    cal.fy_half_year_name       AS fy_half_year_name,
    cjdv.`FN Code2`             AS `FN Code2`,
    cjdv.`FN Name2`             AS `FN Name2`,
    pdv.`03.Category`           AS prod_category,
    pdv.`04.SubCategory (Lang)` AS subcat_lang,
    pdv.`76.Overlapping SKU (Electro)` AS overlapping_sku,
    cjdv.`Zakka Flag2`          AS zakka_flag2,
    cjdv.`Gillette Flag2`       AS gillette_flag2,
    wdv.`Customer Parent Group` AS wh_parent_group,
    SUM(CAST(isdfv.gross_transact_amt AS DECIMAL(38,6))) AS giv,
    SUM(CAST(isdfv.net_transact_amt   AS DECIMAL(38,6))) AS niv_calc
  FROM hive_metastore.japan_linkedexcel.indirect_ship_day_fct_vw isdfv
  JOIN hive_metastore.japan_linkedexcel.customer_japan_dim_vw cjdv
    ON cjdv.`Cust Key` = isdfv.cust_key
  JOIN hive_metastore.japan_linkedexcel.product_dim_vw pdv
    ON pdv.prod_key = isdfv.prod_key
  JOIN hive_metastore.japan_linkedexcel.calender_dim_vw cal
    ON isdfv.day_date = cal.`Day Date (Real)`
  LEFT JOIN hive_metastore.japan_linkedexcel.warehouse_dim_vw wdv
    ON isdfv.whkey = wdv.cust_key
  WHERE cal.`PG Year (Real)` = 'FY2425'
    AND cal.fy_half_year_name = '2H'
    AND cjdv.`FN Code2` IS NOT NULL
    AND cjdv.`Customer Parent Group` = 'RT'
  GROUP BY
    cal.`PG Year (Real)`,
    cal.`PG Quarter (Real)`,
    cal.fy_half_year_name,
    cjdv.`FN Code2`,
    cjdv.`FN Name2`,
    pdv.`03.Category`,
    pdv.`04.SubCategory (Lang)`,
    pdv.`76.Overlapping SKU (Electro)`,
    cjdv.`Zakka Flag2`,
    cjdv.`Gillette Flag2`,
    wdv.`Customer Parent Group`
)

SELECT
  fd.`PG Year (Real)`    AS FY,
  fd.fy_half_year_name   AS Period,
  fd.`PG Quarter (Real)` AS Quarter,
  fd.`FN Code2`          AS `FN Code`,
  fd.`FN Name2`          AS `FN Name`,
  fd.Category,

  CAST(
    SUM(
      CASE
        WHEN fd.Category IN ('Laundry','Fabric Enhancer','Dish Care','Air Care','Baby Care','Feminine Care','Hair Care')
         AND fu.prod_category = fd.Category
         AND fu.subcat_lang <> '生理用品'
         AND fu.zakka_flag2 = 'Z-DC'
        THEN fu.giv
        WHEN fd.Category = 'Shave Care'
         AND fu.prod_category = 'Shave Care'
         AND fu.gillette_flag2 = 'G-DC'
        THEN fu.giv
        WHEN fd.Category IN ('Oral Care','Appliances')
         AND fu.src = 'IND'
         AND fu.prod_category = fd.Category
         AND fu.gillette_flag2 = 'G-DC'
         AND COALESCE(fu.wh_parent_group,'') <> 'WS Electro'
        THEN fu.giv
        WHEN fd.Category IN ('Oral Care','Appliances')
         AND fu.src = 'SHIP'
         AND fu.prod_category = fd.Category
         AND fu.gillette_flag2 = 'G-DC'
         AND fu.overlapping_sku IN ('Zakka-Oral','Zakka-Appliances')
        THEN fu.giv
        ELSE 0
      END
    ) AS BIGINT
  ) AS `Actual GIV`,

  CAST(
    SUM(
      CASE
        WHEN fd.Category IN ('Laundry','Fabric Enhancer','Dish Care','Air Care','Baby Care','Feminine Care','Hair Care')
         AND fu.prod_category = fd.Category
         AND fu.subcat_lang <> '生理用品'
         AND fu.zakka_flag2 = 'Z-DC'
        THEN fu.niv_calc
        WHEN fd.Category = 'Shave Care'
         AND fu.prod_category = 'Shave Care'
         AND fu.gillette_flag2 = 'G-DC'
        THEN fu.niv_calc
        WHEN fd.Category IN ('Oral Care','Appliances')
         AND fu.src = 'IND'
         AND fu.prod_category = fd.Category
         AND fu.gillette_flag2 = 'G-DC'
         AND COALESCE(fu.wh_parent_group,'') <> 'WS Electro'
        THEN fu.niv_calc
        WHEN fd.Category IN ('Oral Care','Appliances')
         AND fu.src = 'SHIP'
         AND fu.prod_category = fd.Category
         AND fu.gillette_flag2 = 'G-DC'
         AND fu.overlapping_sku IN ('Zakka-Oral','Zakka-Appliances')
        THEN fu.niv_calc
        ELSE 0
      END
    ) AS BIGINT
  ) AS `Actual NIV`

FROM fd
LEFT JOIN fact_union fu
  ON fu.`PG Year (Real)`     = fd.`PG Year (Real)`
 AND fu.`PG Quarter (Real)`  = fd.`PG Quarter (Real)`
 AND fu.fy_half_year_name    = fd.fy_half_year_name
 AND fu.`FN Code2`           = fd.`FN Code2`
 AND fu.prod_category        = fd.Category

GROUP BY
  fd.`PG Year (Real)`,
  fd.fy_half_year_name,
  fd.`PG Quarter (Real)`,
  fd.`FN Code2`,
  fd.`FN Name2`,
  fd.Category

ORDER BY
  FY,
  Period,
  Quarter,
  `FN Code`,
  Category;